In [ ]:
from tqdm.notebook import tqdm
import pandas as pd

import sys
sys.path.append('..')
from src.web_scrapping_retail import WebScrappingRetail

In [ ]:
## DataFrame con los datos de scrapping
scraper = WebScrappingRetail(max_por_categoria=100)
productos = scraper.run()

In [ ]:
# Construir DataFrame
print(f'\n{"="*50}')
print(f'Total antes de limpiar: {len(productos):,}')

if len(productos) == 0:
    print('No se extrajo ningún producto.')
else:
    df = pd.DataFrame(productos)
    df = df.drop_duplicates(subset=['nombre', 'precio']).reset_index(drop=True)
    df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
    df = df[['nombre', 'departamento', 'categoria', 'subcategoria',
             'marca', 'contenido', 'precio', 'descripcion']]

    print(f'{len(df):,} productos únicos')
    print(f'\nCobertura:')
    print(f'  Con marca    : {df["marca"].ne("").sum():,} ({df["marca"].ne("").mean()*100:.1f}%)')
    print(f'  Con contenido: {df["contenido"].notna().sum():,} ({df["contenido"].notna().mean()*100:.1f}%)')
    display(df.head(15))

    # guardar DataFrame raw
    ts      = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
    archivo = f'jumbo_productos_{ts}.csv'    
    df.to_csv(f'../data/raw/{archivo}', index=False)  
 

    # Análisis de cobertura de contenido por categoría 
    print(f'\n{"="*50}')
    print('Cobertura de campo "contenido" por categoría:')

    df['contenido_vacio'] = (
        df['contenido'].isna() |
        df['contenido'].astype(str).str.strip().isin(['', 'None', 'nan'])
    )

    total   = df.groupby('categoria').size().reset_index(name='total')
    vacios  = (df[df['contenido_vacio']]
               .groupby('categoria').size()
               .reset_index(name='vacios'))

    resumen = (total
               .merge(vacios, on='categoria', how='left')
               .fillna(0)
               .assign(vacios=lambda x: x['vacios'].astype(int))
               .assign(pct_vacio=lambda x: (x['vacios'] / x['total'] * 100).round(1))
               .sort_values('pct_vacio', ascending=False)
               .reset_index(drop=True))

    display(resumen)

    # Filtrar categorías no alimentarias
    print(f'\n{"="*50}')
    print('Aplicando filtros de departamentos y categorías...')

    DEPTOS_DESCARTAR = [
        'Ropa Y Accesorios',
        'Especiales',
        'Mascotas',
        'Libros Y Papelería',
        'Juguetería',
    ]

    CATEGORIAS_DESCARTAR = [
        # Hogar
        'Organización de Ropa',
        'Muebles y Accesorios para Baño',
        'Terraza Y Exterior',
        # Herramientas y Ferretería
        'Cerrajería',
        'Grifería para Baño',
        'Grifería para Cocina',
        'Herramientas para Construcción',
        # Mundo Bebés
        'Desarrollo Y Estimulación',
        'Cuarto Del Bebé',
        'Lencería Y Decoración',
        'Paseo Y Viajes',
        # Deportes
        'Otros Deportes',
        'Bicicletas',
        # Pinturas
        'Herramientas Para Pintar',
        'Industrial',
    ]

    df_filtrado = df[
        ~df['departamento'].isin(DEPTOS_DESCARTAR) &
        ~df['categoria'].isin(CATEGORIAS_DESCARTAR) &
        (df['precio'] > 0)
    ].copy().reset_index(drop=True)

    print(f'  Registros originales : {len(df):,}')
    print(f'  Registros después    : {len(df_filtrado):,}')
    print(f'  Eliminados           : {len(df) - len(df_filtrado):,}')
    print(f'\nDepartamentos finales:\n{df_filtrado["departamento"].value_counts().to_string()}')

## Guardar DataFrame procesado

df_filtrado.to_csv('../data/processed/productos_retail_procesados.csv', index=False)